In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.concat(
    [
        pd.DataFrame(iris.data, columns = iris.feature_names),
        pd.DataFrame(iris.target, columns = ['target'])
    ],
    axis = 1
)

In [ ]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [ ]:
kf = StratifiedKFold(n_splits = 10, shuffle = True, random_state = 42)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(),
    'SVM': SVC(),
    'Random Forest': RandomForestClassifier(random_state = 42)
}

params = {
    'Logistic Regression': {
        'max_iter': [200, 300, 400]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf']
    },
    'Random Forest': {
        'criterion': ['gini', 'entropy'],
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
    }
}

In [ ]:
X = df.drop('target', axis = 1)
y = df['target']

In [ ]:
results = []

for model_name, model in models.items():
    best_score = 0
    best_params = {}

    combinations = [{}]

    for key, values in params[model_name].items():
        new_combinations = []

        for combo in combinations:
            for value in values:
                new_combo = combo.copy()
                new_combo[key] = value

                new_combinations.append(new_combo)

        combinations = new_combinations

    for combo in combinations:
        model.set_params(**combo)

        scores = cross_val_score(model, X, y, cv = kf)
        mean_score = scores.mean()

        if mean_score > best_score:
            best_score = mean_score
            best_params = combo

    results.append({
        'model': model_name,
        'Best Parameters': best_params,
        'Best Score': best_score
    })

In [ ]:
results_df = pd.DataFrame(results)
results_df

,model,Best Parameters,Best Score
0,Logistic Regression,{'max_iter': 200},0.966667
1,SVM,"{'C': 1, 'kernel': 'linear'}",0.980000
2,Random Forest,"{'criterion': 'gini', 'n_estimators': 100, 'ma...",0.953333
